《Think Stats》 3판은 [Bookshop.org](https://bookshop.org/a/98697/9781098190255)와 [Amazon](https://amzn.to/42lmxwu)에서 구할 수 있습니다. 두 링크는 원저자의 제휴 링크입니다. 책이 도움이 되었다면 Green Tea Press의 [무료 공개 교육 자료 제작을 후원](https://greenteapress.com/support/?book=ts3)하는 것도 생각해 주세요.

# 최소제곱

이 장과 다음 장에서는 자료에 모형을 맞추는 방법을 다룹니다. 여기서 **모형**은 직선처럼 변수들의 관계를 나타내는 수학적 표현과, 정규분포처럼 무작위 변동을 나타내는 설명으로 이루어집니다.

모형이 자료에 잘 맞는다는 말은 보통 모형과 관측값 사이의 거리를 나타내는 오차가 작다는 뜻입니다. 널리 쓰이는 방법 중 하나는 오차 제곱의 합을 최소화하는 **최소제곱**입니다. 이 장에서는 변수 두 개를 다루고, 다음 장에서는 여러 설명변수를 사용하는 모형으로 나아가겠습니다.

[원문 노트북을 Colab에서 실행하기](https://colab.research.google.com/github/AllenDowney/ThinkStats/blob/30ecbdcf05723f1190cae2aeaff5848f59f8ca26/nb/chap10.ipynb).

In [1]:
from os.path import basename, exists


def download(url):
    filename = basename(url)
    if not exists(filename):
        from urllib.request import urlretrieve

        local, _ = urlretrieve(url, filename)
        print("Downloaded " + local)


download("https://github.com/AllenDowney/ThinkStats/raw/v3/nb/thinkstats.py")

In [2]:
try:
    import empiricaldist
except ImportError:
    %pip install empiricaldist

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from thinkstats import decorate

## 최소제곱 직선

[8장](/posts/statistics/think-stats-08#펭귄의-체중-재기)의 연구 상황으로 돌아가겠습니다. 남극의 펭귄을 잠시 포획해 크기와 체중을 측정하고 안전하게 놓아줍니다. 그런데 펭귄이 저울 위에 오래 머무르지 않아 정확한 체중을 재기 어려울 수 있습니다. 날개나 부리 크기는 알지만 체중은 없는 경우에 다른 측정값으로 체중을 추정할 수 있을까요? 빠진 값을 추정해 채우는 작업을 **대치**(imputation)라고 합니다.

2007~2010년 남극 팔머 기지 연구진이 수집한 공개 자료에서 체중과 다른 측정값의 관계를 살펴보겠습니다. 자료를 가져오는 방법은 이 장의 노트북에 있습니다.

자료 출처는 Allison Horst의 저장소입니다. Horst, Hill, Gorman(2020), *palmerpenguins*, [DOI: 10.5281/zenodo.3960218](https://doi.org/10.5281/zenodo.3960218). 원래 연구는 Gorman, Williams, Fraser(2014), *PLOS ONE* 9(3), e90081, [DOI: 10.1371/journal.pone.0090081](https://doi.org/10.1371/journal.pone.0090081)입니다.

In [4]:
download(
    "https://raw.githubusercontent.com/allisonhorst/palmerpenguins/c19a904462482430170bfe2c718775ddb7dbb885/inst/extdata/penguins_raw.csv"
)

`read_csv`로 자료를 읽습니다.

In [5]:
penguins = pd.read_csv("penguins_raw.csv").dropna(subset=["Body Mass (g)"])
penguins.shape

체중이 알려진 아델리펭귄은 151마리입니다. `query`로 해당 행을 고릅니다.

In [6]:
adelie = penguins.query('Species.str.startswith("Adelie")')
len(adelie)

아델리펭귄의 날개 길이를 알 때 체중을 얼마나 잘 예측할 수 있을지 알아보겠습니다. 필요한 두 열을 선택합니다.

In [7]:
xvar = "Flipper Length (mm)"
yvar = "Body Mass (g)"

flipper_length = adelie[xvar]
body_mass = adelie[yvar]

날개 길이와 체중의 관계를 산점도로 그립니다.

In [8]:
plt.scatter(flipper_length, body_mass, marker=".", alpha=0.5)
decorate(xlabel=xvar, ylabel=yvar)

관련이 있어 보입니다. 상관계수로 관계의 강도를 수치화할 수 있습니다.

In [9]:
np.corrcoef(flipper_length, body_mass)[0, 1]

상관은 약 0.47입니다. 날개가 긴 펭귄이 대체로 무겁습니다. 따라서 날개 길이를 알면 체중을 더 잘 추측할 수 있겠지만, 상관계수만으로 구체적인 예측값을 구할 수는 없습니다. **가장 잘 맞는 직선**을 골라야 합니다.

‘잘 맞는다’는 기준은 여러 가지입니다. 여기서는 평균제곱오차(MSE)를 최소화하는 **선형 최소제곱** 직선을 사용하겠습니다. SciPy의 `linregress`가 이 계산을 합니다. 이름은 선형회귀(linear regression)의 줄임말이며, 인수는 x값과 y값 순서입니다.

In [10]:
from scipy.stats import linregress

result = linregress(flipper_length, body_mass)
result

결과인 `LinregressResult`에는 기울기와 절편을 비롯한 정보가 들어 있습니다. 기울기는 약 32.8g/mm입니다. 날개가 1mm 길어질 때 체중의 예측 평균이 약 32.8g 높아진다는 뜻입니다.

절편은 약 −2535g입니다. 음의 체중이라 이상해 보일 수 있지만, 이 값은 날개 길이 0mm에서 직선을 연장한 결과입니다. 실제 자료의 평균 날개 길이에서 예측값을 구하면 뜻을 더 쉽게 읽을 수 있습니다.

In [11]:
x = flipper_length.mean()
y = result.intercept + result.slope * x
x, y

평균에 해당하는 약 190mm 길이에서는 예측 체중이 약 3700g입니다. 다음 함수는 회귀 결과와 x값들을 받아 직선 위의 y값들을 계산합니다.

In [12]:
def predict(result, xs):
    ys = result.intercept + result.slope * xs
    return ys

일상에서 ‘예측’은 보통 미래를 가리키지만, 회귀에서는 적합한 직선 위의 값도 **예측값**이라고 부릅니다. 여러 날개 길이에 대응하는 예측값을 구하겠습니다.

In [13]:
fit_xs = np.linspace(np.min(flipper_length), np.max(flipper_length))
fit_ys = predict(result, fit_xs)

회귀선을 산점도 위에 함께 그립니다.

In [14]:
plt.scatter(flipper_length, body_mass, marker=".", alpha=0.5)
plt.plot(fit_xs, fit_ys, color="C1")
decorate(xlabel=xvar, ylabel=yvar)

직선은 자료의 가운데를 지나며 전반적인 경향을 따릅니다. 하지만 많은 관측점이 직선에서 멀리 있습니다. 예측이 얼마나 정확한지 보려면 관측값에서 같은 x의 예측값까지 세로 방향의 차이를 계산하면 됩니다. 이 차이를 **잔차**라고 부릅니다.

In [15]:
def compute_residuals(result, xs, ys):
    fit_ys = predict(result, xs)
    return ys - fit_ys

날개 길이로 체중을 예측했을 때의 잔차를 구합니다.

In [16]:
residuals = compute_residuals(result, flipper_length, body_mass)

자료의 첫 번째 펭귄을 예로 보겠습니다.

In [17]:
x = flipper_length[0]
y = predict(result, x)
x, y

이 펭귄의 날개 길이는 181mm이고 예측 체중은 약 3407g입니다. 실제 체중과 비교하겠습니다.

In [18]:
body_mass[0], residuals[0]

실제 체중은 3750g이므로 잔차는 약 343g입니다. 잔차를 제곱해 평균 내면 **평균제곱오차**(MSE)가 됩니다.

In [19]:
mse = np.mean(residuals**2)
mse

이 숫자만으로는 예측의 질을 판단하기 어렵습니다. 결정계수를 구하면 더 해석하기 쉽습니다.

## 결정계수

날개 길이를 알면 최소제곱 회귀선으로 체중을 추측하고, MSE로 평균적인 오차를 나타낼 수 있습니다. 길이를 모른다면 무엇을 예측해야 할까요? 항상 체중의 표본평균을 답하는 것이 MSE를 최소화합니다. 이때 예측 오차는 평균에서의 편차입니다.

In [20]:
deviations = body_mass - np.mean(body_mass)

이 예측의 MSE는 평균에서의 편차를 제곱한 평균입니다.

In [21]:
np.mean(deviations**2)

바로 분산의 정의와 같습니다.

In [22]:
np.var(body_mass)

체중의 분산은 평균만 예측했을 때의 MSE입니다. 여기서는 절편을 포함해 직선을 맞췄으므로 잔차의 평균이 0이고, 잔차의 분산도 회귀선을 사용했을 때의 MSE와 같습니다. 두 값의 비를 1에서 빼면 날개 길이 정보가 MSE를 얼마나 줄였는지 알 수 있습니다.

이를 **결정계수**라고 부르고 $R^2$로 씁니다. 보통 ‘알 제곱’이라고 읽습니다.

In [23]:
def coefficient_of_determination(ys, residuals):
    return 1 - np.var(residuals) / np.var(ys)

이 예의 $R^2$는 약 0.22입니다. 평균만 예측할 때에 비해 회귀선의 훈련 자료 MSE가 약 22% 작다는 뜻입니다.

In [24]:
R2 = coefficient_of_determination(body_mass, residuals)
R2

절편을 포함한 단순 선형 최소제곱에서는 상관계수와 결정계수 사이에 $R^2=r^2$의 관계가 있습니다. $R^2$의 제곱근을 계산해 보겠습니다.

In [25]:
r = np.sqrt(R2)
r

앞서 구한 양의 상관계수와 비교합니다.

In [26]:
corr = np.corrcoef(flipper_length, body_mass)[0, 1]
corr

부동소수점 근사에 따른 작은 차이를 제외하면 같습니다. `linregress`도 이 값을 결과의 속성으로 제공합니다.

In [27]:
result.rvalue

두 통계량은 관련되어 있지만 읽는 방식이 다릅니다. 상관계수는 선형 관계의 방향과 강도를 −1~1 사이의 값으로 나타냅니다. $R^2$는 평균만 예측할 때에 비해 MSE가 얼마나 줄었는지를 나타냅니다. 이 예에서는 상관계수를 제곱하므로 방향 정보는 사라집니다.

> 번역자 주: $\sqrt{R^2}=|r|$이므로 음의 상관의 부호까지 복원되지는 않습니다. $R^2=r^2$는 절편이 있는 단순 최소제곱의 훈련 자료에 대한 성질입니다. 절편이 없는 모형이나 별도 평가 자료에서 구한 $R^2$는 음수가 될 수도 있습니다. 22%를 ‘체중의 22%를 맞힌다’는 뜻으로 해석하지 마세요.

## MSE 최소화하기

최소제곱 직선이 MSE를 최소화한다는 것을 증명하는 대신 실험으로 확인하겠습니다. 기울기와 절편에 작은 난수를 더하면 MSE가 더 커지는지 보겠습니다.

In [28]:
intercept = result.intercept + np.random.normal(0, 1)
slope = result.slope + np.random.normal(0, 1)

수정한 기울기와 절편을 담을 객체로 `types` 모듈의 `SimpleNamespace`를 사용합니다.

In [29]:
from types import SimpleNamespace

fake_result = SimpleNamespace(intercept=intercept, slope=slope)
fake_result

이 객체를 `compute_residuals`에 전달해 잔차와 MSE를 구합니다.

In [30]:
fake_residuals = compute_residuals(fake_result, flipper_length, body_mass)
fake_mse = np.mean(fake_residuals**2)

최소제곱 결과와 비교하면, 무작위로 옮긴 직선의 MSE가 더 큽니다.

In [31]:
mse, fake_mse, fake_mse > mse

하지만 MSE를 최소화하는 것만이 ‘최선’의 기준은 아닙니다. 잔차 절댓값의 합이나 관측점에서 직선까지 거리의 합을 최소화할 수도 있습니다. 과대예측과 과소예측의 비용이 다르다면, 각 잔차에 비용을 매겨 전체 비용을 줄이는 편이 맞을 수 있습니다.

최소제곱은 계산이 효율적이라 널리 쓰입니다. 다음 함수가 그 계산을 보여 줍니다.

In [32]:
def least_squares(xs, ys):
    xbar = np.mean(xs)
    ybar = np.mean(ys)

    xdev = xs - xbar
    ydev = ys - ybar

    slope = np.sum(xdev * ydev) / np.sum(xdev**2)
    intercept = ybar - slope * xbar

    return intercept, slope

날개 길이와 체중으로 이 함수를 시험하겠습니다.

In [33]:
intercept, slope = least_squares(flipper_length, body_mass)
intercept, slope

`linregress`와 같은 절편·기울기를 얻는지 확인할 수 있습니다.

In [34]:
np.allclose([intercept, slope], [result.intercept, result.slope])

계산 비용 때문에 방법을 골라야 했던 때와 달리, 지금은 문제에 맞는 오차 기준을 선택할 여지가 큽니다. 잔차의 제곱을 최소화하는 것이 현재 문제에 적절한지도 생각해야 합니다.

## 추정

기울기와 절편은 표본에서 구한 추정값입니다. 대표성 없는 표본, 측정 오류, 무작위 표집 변동의 영향을 받습니다. 앞의 두 원인을 수치화하기는 어렵지만 표집 변동은 비교적 다루기 쉽습니다.

한 방법은 **부트스트랩 재표집**입니다. 관측 표본을 모집단처럼 보고, 원래 표본과 같은 크기로 복원추출합니다. 다음 함수는 `DataFrame`의 행을 다시 뽑습니다.

In [35]:
def resample(df):
    n = len(df)
    return df.sample(n, replace=True)

다음 함수는 재표집한 자료에서 최소제곱 회귀선을 구하고 기울기를 반환합니다.

In [36]:
def estimate_slope(df):
    xs, ys = df["Flipper Length (mm)"], df["Body Mass (g)"]
    result = linregress(xs, ys)
    return result.slope

두 함수를 결합하면 가상 표본을 여러 개 만들고 각각의 기울기를 구할 수 있습니다.

In [37]:
# Seed the random number generator so we get the same results every time

np.random.seed(1)

In [38]:
resampled_slopes = [estimate_slope(resample(adelie)) for i in range(1001)]

이 기울기들은 기울기 추정량의 표집분포를 근사합니다. 분포를 그려 보겠습니다.

In [39]:
from thinkstats import plot_kde

plot_kde(resampled_slopes)
decorate(xlabel="Slope of the fitted line (g / mm)", ylabel="Density")

백분위수로 90% 신뢰구간을 구할 수 있습니다.

In [40]:
ci90 = np.percentile(resampled_slopes, [5, 95])
print(result.slope, ci90)

원문의 결과는 기울기 약 33g/mm, 90% 신뢰구간 약 [25, 40]g/mm로 보고할 수 있습니다. 추정량의 표준오차는 이 재표집 분포의 표준편차입니다.

In [41]:
stderr = np.std(resampled_slopes)
stderr

`linregress` 결과에도 분포에 대한 가정을 바탕으로 계산한 표준오차 근삿값이 들어 있습니다.

In [42]:
result.stderr

이 예에서 재표집으로 얻은 표준오차가 조금 작지만, 실질적으로 큰 차이는 아닙니다.

## 불확실성 시각화하기

재표집할 때마다 회귀선이 조금 달라집니다. 얼마나 달라지는지 보려면 여러 회귀선을 겹쳐 그릴 수 있습니다. 다음 함수는 재표집한 자료에 직선을 맞추고 지정된 x값들의 예측값을 반환합니다.

In [43]:
def fit_line(df, fit_xs):
    xs, ys = df["Flipper Length (mm)"], df["Body Mass (g)"]
    result = linregress(xs, ys)
    fit_ys = predict(result, fit_xs)
    return fit_ys

평가에 사용할 x값 범위는 다음과 같습니다.

In [44]:
xs = adelie["Flipper Length (mm)"]
fit_xs = np.linspace(np.min(xs), np.max(xs))

관측 자료 위에 여러 회귀선을 겹쳐 그립니다.

In [45]:
plt.scatter(flipper_length, body_mass, marker=".", alpha=0.5)

for i in range(101):
    fit_ys = fit_line(resample(adelie), fit_xs)
    plt.plot(fit_xs, fit_ys, color="C1", alpha=0.05)

decorate(xlabel=xvar, ylabel=yvar)

자료의 가운데에서는 선들이 모이고 양 끝에서는 더 퍼집니다. 각 x에서 예측 평균의 90% 신뢰구간을 그리는 방법도 있습니다. 회귀선별 예측 배열들을 모으겠습니다.

In [46]:
fitted_ys = [fit_line(resample(adelie), fit_xs) for i in range(1001)]

이 리스트를 2차원 배열로 보면 행은 회귀선이고 열은 x값입니다. `axis=0`으로 각 열의 5·50·95백분위수를 구합니다.

In [47]:
low, median, high = np.percentile(fitted_ys, [5, 50, 95], axis=0)

`fill_between`으로 5~95백분위수 사이를 채우고, 가운데 값과 원자료도 함께 그립니다.

In [48]:
plt.scatter(flipper_length, body_mass, marker=".", alpha=0.5)

plt.fill_between(fit_xs, low, high, color="C1", lw=0, alpha=0.2)
plt.plot(fit_xs, median, color="C1")

decorate(xlabel=xvar, ylabel=yvar)

표집 변동 때문에 회귀선이 얼마나 달라질 수 있는지 보여줄 때 제가 선호하는 방법입니다.

> 번역자 주: 이 띠는 각 x에서 평균 회귀선의 불확실성을 나타냅니다. 새 펭귄 한 마리의 체중을 포함하려는 예측구간은 개체 간 잔차 변동까지 더해야 하므로 보통 더 넓습니다. 부트스트랩은 행들이 적절한 독립 표본이라는 가정 아래 표집 변동을 근사하며, 표집편향이나 빠진 변수 문제를 해결하지는 않습니다.

## 변수 변환

직선을 맞추기 전에 제곱·제곱근·로그처럼 한쪽 또는 양쪽 변수를 변환하면 도움이 될 때가 있습니다. [5장](/posts/statistics/think-stats-05#로그정규분포)에서 소개한 BRFSS 키·체중 자료로 보겠습니다.

BRFSS 원자료를 불러오는 코드는 내려받는 노트북에 있습니다.

In [49]:
download("https://github.com/AllenDowney/ThinkStats/raw/v3/data/CDBRFS08.ASC.gz")

자료를 다음과 같이 읽습니다.

In [50]:
from thinkstats import read_brfss

brfss = read_brfss()

키와 체중이 모두 있는 행만 고르고, 두 열을 선택합니다.

In [51]:
valid = brfss.dropna(subset=["htm3", "wtkg2"])
heights, weights = valid["htm3"], valid["wtkg2"]

`linregress`로 최소제곱 기울기와 절편을 구합니다.

In [52]:
result_brfss = linregress(heights, weights)
result_brfss.intercept, result_brfss.slope

기울기는 약 0.96kg/cm입니다. 키가 1cm 높은 사람은 평균 체중이 거의 1kg 높은 경향을 나타냅니다. 여러 키에 대한 예측값을 계산합니다.

In [53]:
fit_xs = np.linspace(heights.min(), heights.max())
fit_ys = predict(result_brfss, fit_xs)

점이 겹치는 것을 줄이기 위해, 그림에 사용할 키와 체중에는 지터를 더하겠습니다.

In [54]:
from thinkstats import jitter

jittered_heights = jitter(heights, 2)
jittered_weights = jitter(weights, 1.5)

키의 평균과 표준편차로 x축 범위를 정합니다.

In [55]:
m, s = heights.mean(), heights.std()
xlim = m - 4 * s, m + 4 * s
ylim = 0, 200

지터를 준 자료와 회귀선을 함께 그립니다.

In [56]:
plt.scatter(jittered_heights, jittered_weights, alpha=0.01, s=0.1)
plt.plot(fit_xs, fit_ys, color="C1")
decorate(xlabel="Height (cm)", ylabel="Weight (kg)", xlim=xlim, ylim=ylim)

회귀선이 점들이 가장 밀집한 부분을 지나지 않습니다. 성인 체중은 정규분포보다 오른쪽으로 치우친 로그정규분포에 가깝고, 큰 체중들이 직선을 위로 끌어당기기 때문입니다. 잔차 분포도 살펴보겠습니다.

In [57]:
residuals = compute_residuals(result_brfss, heights, weights)

In [58]:
from thinkstats import make_pmf

pmf_kde = make_pmf(residuals, -60, 120)
pmf_kde.plot()

decorate(xlabel="Residual (kg)", ylabel="Density")

잔차 분포는 오른쪽으로 치우쳐 있습니다. 그것만으로 곧바로 잘못된 모형이라는 뜻은 아니지만, 현재 직선이 관계의 모양을 충분히 담지 못했을 수 있습니다. 체중이 로그정규분포에 가깝다면 로그 체중은 정규분포에 가까우므로, 이번에는 키로 로그 체중을 설명하는 직선을 맞추겠습니다.

In [59]:
log_weights = np.log10(weights)
result_brfss2 = linregress(heights, log_weights)
result_brfss2.intercept, result_brfss2.slope

변환한 변수의 기울기와 절편은 해석하기 어려워집니다. 우선 그 직선의 예측값을 계산합니다.

In [60]:
fit_xs = np.linspace(heights.min(), heights.max())
fit_ys = predict(result_brfss2, fit_xs)

변환한 자료의 산점도와 함께 그립니다.

In [61]:
jittered_log_weights = jitter(log_weights, 1.5)
plt.scatter(jittered_heights, jittered_log_weights, alpha=0.01, s=0.1)
plt.plot(fit_xs, fit_ys, color="C1")
decorate(xlabel="Height (cm)", ylabel="Weight (log10 kg)", xlim=xlim)

이번 회귀선은 점들의 가운데를 지나며 위아래 퍼짐도 더 대칭적으로 보입니다.

In [62]:
residuals = compute_residuals(result_brfss2, heights, log_weights)

In [63]:
pmf_kde = make_pmf(residuals, -0.6, 0.6)
pmf_kde.plot()

decorate(xlabel="Residual (kg)", ylabel="Density")

산점도와 잔차 모양은 키와 로그 체중의 관계가 직선으로 더 잘 표현된다는 것을 보여 줍니다. 로그 변환 후 상관도 조금 더 큽니다.

In [64]:
result_brfss.rvalue, result_brfss2.rvalue

따라서 이 단순회귀의 $R^2$도 조금 더 큽니다.

In [65]:
result_brfss.rvalue**2, result_brfss2.rvalue**2

로그 체중을 사용하면 이 자료의 관계가 직선으로 조금 더 잘 표현됩니다. 다만 변수 변환은 계수의 해석을 어렵게 하므로, 결과를 제시할 때 역변환이 도움이 됩니다. 상용로그의 역변환은 10의 거듭제곱입니다. 원래 단위로 돌아온 곡선을 원자료와 함께 그리겠습니다.

In [66]:
plt.scatter(jittered_heights, jittered_weights, alpha=0.01, s=0.1)
plt.plot(fit_xs, 10**fit_ys, color="C1")
decorate(xlabel="Height (cm)", ylabel="Weight (kg)", xlim=xlim, ylim=ylim)

로그 체중에 대해 직선이었던 모형은 원래 체중 단위에서는 곡선이 됩니다.

> 번역자 주: 로그 모형의 잔차 단위는 kg가 아니라 log10(kg)입니다. 또 로그 평균의 역변환은 일반적으로 원래 체중의 조건부 평균과 같지 않습니다. 로그정규 오차라면 보정이 필요합니다. 변환 전후 $R^2$는 서로 다른 척도의 오차를 비교하므로 원래 단위의 예측 성능까지 자동으로 우열이 정해지지는 않습니다.

## 용어 정리

- **모형**: 변수 관계의 수학적 표현과 무작위 변동에 대한 설명.
- **대치**: 빠진 자료를 추정해 채우는 과정.
- **최적 적합선**: 정한 오차 기준에서 자료의 관계를 가장 잘 설명하는 선이나 곡선.
- **선형회귀**: 직선 모형의 계수를 추정하는 방법.
- **예측값**: 적합선 위의 값. 반드시 미래에 관한 예측이라는 뜻은 아닙니다.
- **잔차**: 관측값에서 예측값을 뺀 차이.
- **선형 최소제곱**: 잔차 제곱의 합을 최소화하는 직선 적합.
- **결정계수**: 평균만 예측하는 기준과 비교해 잔차제곱합이 얼마나 줄었는지 나타내는 $R^2$.
- **부트스트랩 재표집**: 관측 표본에서 같은 크기의 표본을 복원추출하는 방법.

## 연습문제

### 연습문제 10.1

이 장에서는 날개 길이로 아델리펭귄의 체중을 예측했습니다. 부리 길이와 깊이도 사용할 수 있습니다. 부리 길이를 설명변수로 최소제곱 회귀를 하고 산점도와 회귀선을 그리세요. `rvalue`와 결정계수는 얼마인가요? 날개 길이와 부리 길이 중 어느 쪽의 훈련 자료 예측력이 더 좋은가요?

In [67]:
xvar = "Culmen Length (mm)"
yvar = "Body Mass (g)"

culmen_length = adelie[xvar]
body_mass = adelie[yvar]

In [68]:
# Solution goes here

In [69]:
# Solution goes here

In [70]:
# Solution goes here

In [71]:
# Solution goes here

### 연습문제 10.2

절편의 표집분포도 재표집으로 근사할 수 있습니다.

1. 재표집한 `DataFrame`을 받아 체중을 날개 길이로 회귀하고 절편을 반환하는 `estimate_intercept`를 작성하세요.
2. `adelie`를 여러 번 재표집해 절편을 모으세요.
3. `plot_kde`로 분포를 그리세요.
4. 표준오차와 90% 신뢰구간을 구하세요.
5. `intercept_stderr`와 비교하세요. 재표집 추정치가 조금 작을 수 있습니다.

In [72]:
# Solution goes here

In [73]:
# Solution goes here

In [74]:
# Solution goes here

In [75]:
# Solution goes here

In [76]:
# Solution goes here

### 연습문제 10.3

체질량지수(BMI)는 kg 단위 체중을 m 단위 키의 제곱으로 나눈 값입니다. 키를 cm에서 m로 바꾸어 계산합니다.

In [77]:
heights_m = heights / 100
bmis = weights / heights_m**2

정의에 키의 제곱이 들어가는 것은 평균 체중이 대략 키의 제곱에 비례한다는 통계학 초기의 관찰에 바탕을 둡니다. BRFSS 자료와 회귀로 이를 살펴보세요.

체중이 키의 어떤 거듭제곱 $a$에 비례한다고 가정하면

$$w=bh^a$$

입니다. $w$는 체중, $h$는 키, $b$는 비례상수입니다. 양변에 로그를 취하면

$$\log w=\log b+a\log h$$

가 됩니다. 따라서 로그 키로 로그 체중을 회귀한 기울기가 $a$를 추정합니다. 두 변수에 같은 밑의 로그를 사용하세요. 추정 기울기는 2에 가까운가요?

In [78]:
# Solution goes here

In [79]:
# Solution goes here

In [80]:
# Solution goes here

In [81]:
# Solution goes here

In [82]:
# Solution goes here

[Think Stats: Exploratory Data Analysis in Python, 3rd Edition](https://allendowney.github.io/ThinkStats/index.html)

Copyright 2024 [Allen B. Downey](https://allendowney.com)

코드 라이선스: [MIT](https://mit-license.org/)

본문·그림과 한국어 번역 라이선스: [CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)